# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane:** Ranking Signal Analysis — which safe content and search signals are associated with a page's visibility declining, and can that association support a ranked review queue.

**Question:** among FlyRank's anonymized content-refresh data, which observable, pre-decision signals (visibility consistency, position, engagement, staleness) are associated with a page's `trend_direction` moving to `down`, and does a model built on those signals rank pages well enough to be worth a reviewer's time over a hand-written rule?

**Decision it supports:** which existing pages a content reviewer opens first this cycle. **Who acts on it:** a content strategist/editor. **Cost of a wrong call:** review time spent on a page that wasn't actually declining (false positive), or a genuinely declining page never surfaced at all (false negative) — both costs are measured directly in §5 (Results) and §6 (Limitations), not asserted.

In [1]:
!git clone https://github.com/HasnainRaza16/flyrank-ml-Hasnain.git
%cd flyrank-ml-Hasnain


Cloning into 'flyrank-ml-Hasnain'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 191 (delta 85), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (191/191), 1.92 MiB | 16.77 MiB/s, done.
Resolving deltas: 100% (85/85), done.
/content/flyrank-ml-Hasnain


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release used:** the starter dataset shipped in this repo — `data/raw/content_refresh_anonymized.csv` — not the 79M-row Hugging Face warehouse. 30,000 rows, 44 columns, one row per pseudonymized `content_id` (verified unique — zero duplicates), spread unevenly across 32 pseudonymized `client_id`s (as few as 3 rows, as many as ~7,000 per client — confirmed in ML-04, which is why every split in this project groups by client, never samples rows independently).

**Time window:** this file has no `report_date` — it's a single snapshot per item summarizing a trailing 90-day window, plus a nested last-30-days-vs-previous-30-days comparison inside that window. It supports "is this item currently declining," not "when did the decline start" or "how fast is it moving."

**Excluded, with why:**
- `trend_direction`, `trend_pct` — these define the label; using them as features would be leakage by definition.
- `impressions_last_30d`, `impressions_prev_30d` (and clicks/sessions equivalents) — `trend_pct` is computed almost exactly from these (correlation 1.000, verified in ML-04); including them lets the model reconstruct the label through arithmetic, not genuine signal.
- `content_id`, `client_id` — pseudonymous, grouping/splitting only, never model inputs.
- `provider_used` (71.5% missing), `model_used` (19.1% missing) — internal tooling metadata, not a search-performance signal.
- Pre-built `*_tier` bucket columns — bucketed duplicates of raw numerics already in the feature set; keeping both double-counts the same signal.

**Public-safety note:** no client names, URLs, domains, or raw queries appear anywhere in this project — verified by construction, since the shipped file itself is pre-anonymized to pseudonymous IDs only.

In [2]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("rows, cols:", df.shape)
print("unique content_id:", df["content_id"].nunique(), "(grain check: should equal row count)")
print("unique client_id:", df["client_id"].nunique())
print("rows per client — min/median/max:",
      df.groupby("client_id").size().min(), df.groupby("client_id").size().median(), df.groupby("client_id").size().max())
print("decline base rate:", round(df["is_declining_label"].mean(), 3))
print("rows with no usable label (trend_pct missing):", df["trend_pct"].isnull().sum(),
      f"({df['trend_pct'].isnull().mean()*100:.1f}%)")


rows, cols: (30000, 45)
unique content_id: 30000 (grain check: should equal row count)
unique client_id: 32
rows per client — min/median/max: 3 567.0 7008
decline base rate: 0.542
rows with no usable label (trend_pct missing): 3388 (11.3%)


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining_label = (trend_direction == "down")` — an observed, already-computed outcome, not a rule I invented and not a forecast.

**Baseline (ML-07):** a transparent hand rule — flag a page if it's `stale` (no update in 90+ days) AND `visible` (≥500 impressions in 90 days) AND has a `ctr_gap` (CTR below its position tier's median). Staleness alone did not hold up as a signal on its own (checked directly against the label — decline rate stayed close to the base rate across freshness tiers, with the two well-sampled tiers essentially flat); the CTR-vs-position signal did hold up, so the rule gates on both rather than trusting staleness alone.

**Model (ML-08):** Logistic Regression and Random Forest, both trained on the same 23-feature set (current-window engagement/consistency/context signals — no label-window totals, no IDs, no product-tooling flags). Permutation importance run on the model that actually won the precision@K comparison, since the research question is "which signals," not just "can something predict this."

**Validation design (ML-08/ML-09):** `GroupShuffleSplit` on `client_id`, 25% held out, every client entirely in train or test, never both. ML-09 confirmed this matters: the same model scored under a naive random row-level split reports 0.693 ROC AUC; the honest, client-held-out number is 0.589 — a 0.10 gap that a random split would have hidden.

**Leakage checks (ML-04, ML-09):** (1) planted `trend_pct` as a feature on purpose — AUC jumped from an honest 0.599 to a leaked 1.000, confirming the label-derived-feature trap is real and removing it is necessary, not cosmetic. (2) tested whether the 90-day window totals arithmetically overlapping the label's 30-day windows were doing leaky work — removing all eight suspect columns changed ROC AUC by 0.001, a near-zero collapse, so they were kept but the check is logged rather than assumed.

In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
categorical_features = ["content_type", "main_intent", "competition_level"]

df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
numeric_features += ["has_keyword_data", "has_word_count"]

df[numeric_features] = df[numeric_features].fillna(0)
df[categorical_features] = df[categorical_features].fillna("unknown")

X = pd.get_dummies(df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

logreg = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train, y_train)

logreg_proba = logreg.predict_proba(X_test_s)[:, 1]
rf_proba = rf.predict_proba(X_test)[:, 1]

print("ROC AUC — LogReg:", round(roc_auc_score(y_test, logreg_proba), 3))
print("ROC AUC — Random Forest:", round(roc_auc_score(y_test, rf_proba), 3))
print("test clients:", df.iloc[test_idx]["client_id"].nunique(), "| test base rate:", round(y_test.mean(), 3))


ROC AUC — LogReg: 0.589
ROC AUC — Random Forest: 0.604
test clients: 8 | test base rate: 0.517


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**The headline tension:** Random Forest has the higher ROC AUC (0.604 vs 0.589), but on the metric the decision actually needs — precision@K, i.e. "of the top K flagged pages, how many really are declining" — it performs no better than the hand-written baseline. Logistic Regression, despite the lower AUC, wins clearly on precision@K at every K tested. A higher discrimination score does not automatically mean a better ranked queue.

In [4]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Recreate the ML-07 baseline rule, scored on this exact test split
tier_median_ctr = df.loc[df["position_tier"] != "no_data"].groupby("position_tier")["ctr"].median()
df["stale"] = (df["days_since_last_update"] >= 90).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr).fillna(np.inf)
df["ctr_gap"] = ((df["position_tier"] != "no_data") & (df["ctr"] < df["tier_median_ctr"])).astype(int)
df["baseline_score"] = df["stale"] * df["visible"] * df["ctr_gap"] * df["impressions_90d"]
baseline_test = df.iloc[test_idx]["baseline_score"].values

rows = []
for k in [20, 50, 100, 200]:
    rows.append({
        "K": k,
        "baseline_precision@K": round(precision_at_k(baseline_test, y_test.values, k), 3),
        "logreg_precision@K": round(precision_at_k(logreg_proba, y_test.values, k), 3),
        "rf_precision@K": round(precision_at_k(rf_proba, y_test.values, k), 3),
    })
results_table = pd.DataFrame(rows)
results_table["base_rate"] = round(y_test.mean(), 3)
print(results_table.to_string(index=False))


  K  baseline_precision@K  logreg_precision@K  rf_precision@K  base_rate
 20                 0.500               0.850           0.500      0.517
 50                 0.520               0.800           0.520      0.517
100                 0.530               0.740           0.500      0.517
200                 0.545               0.705           0.545      0.517


## 5. Limitations

*What this work cannot claim.*

- **Observed, not forecast.** The label is a computed past outcome. This ranks pages by an already-observed decline; it does not predict what a page will do next.
- **Modest lift, honestly measured.** 0.589 ROC AUC against a 0.517 base rate, on clients never seen in training. The random-split number for this same model (0.693) is 0.10 higher and is not the number to trust or repeat.
- **Validated on 8 held-out clients, from 32 total.** A client whose content mix or scale looks very different from these 32 is out of validated scope until re-checked.
- **Two demonstrated failure modes, not hypothetical ones:** (1) impressions-without-clicks pages get flagged as declining when the real issue may be a CTR/snippet problem; (2) large, well-positioned pages that are actually sliding can still look "healthy" on raw volume and get missed — the model reads scale, not direction of change.
- **Single snapshot, not a time series.** No `report_date` in this file means no way to see when a decline started or how fast it's moving — only whether it's currently flagged.
- **This is the 30k-row starter slice, not the 79M-row warehouse** — patterns here may not hold at the full release's scale or client diversity, and that has not been tested in this project.

In [5]:
# The one sentence this project could be tempted to write, and the honest rewrite (ML-09):
overclaim = "The model predicts which pages will decline."
honest = ("On pages from clients the model has not seen during training, the model's ranking is directionally "
          "better than random at surfacing pages with an already-observed decline in impressions "
          "(ROC AUC 0.589 vs a 0.517 base rate, client-held-out split). It is decision-support for prioritizing "
          "which existing pages a reviewer opens first — it does not forecast future decline, and its lift over "
          "guessing is modest, not dramatic.")
print("NOT USED:", overclaim)
print()
print("USED:", honest)


NOT USED: The model predicts which pages will decline.

USED: On pages from clients the model has not seen during training, the model's ranking is directionally better than random at surfacing pages with an already-observed decline in impressions (ROC AUC 0.589 vs a 0.517 base rate, client-held-out split). It is decision-support for prioritizing which existing pages a reviewer opens first — it does not forecast future decline, and its lift over guessing is modest, not dramatic.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

From the ML-10 action playbook: the queue is capped at the top 200 rows by `pred_proba` — the deepest K actually validated (precision@200 = 0.705). Four archetypes, built from the error analysis above, not from a separate clustering step:

| Archetype | n (of 7,115 test rows) | Suggested action |
|---|---|---|
| `consistent_fade` | 81 | Refresh content — update facts, re-check target keyword. |
| `visibility_without_clicks` | 119 | Check title/meta/snippet first — likely a CTR problem, not a decline problem. |
| `large_page_blindspot` | 523 | Spot-check manually on a rolling basis despite a low score — matches the demonstrated false-negative profile. |
| `stable_low_priority` | 6,392 | No action this cycle. |

**No-go list (never automated from this model alone):** no auto-delete/de-index, no auto-publish of rewrites without a human read, no external claim that the model "predicts" decline, no scoring of a client type outside the 32 validated ones without flagging it as unvalidated.

In [6]:
queue = df.iloc[test_idx][["content_id", "content_type", "main_intent", "days_with_impressions",
                            "days_with_sessions", "users_90d", "avg_position"]].copy().reset_index(drop=True)
queue["pred_proba"] = logreg_proba
queue["consistency_gap"] = queue["days_with_impressions"] - queue["days_with_sessions"]
queue = queue.sort_values("pred_proba", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

TOP_K = 200
GAP_P75 = queue["consistency_gap"].quantile(0.75)
SCALE_P90 = queue["users_90d"].quantile(0.90)

def classify(row):
    if row["rank"] <= TOP_K:
        return "visibility_without_clicks" if row["consistency_gap"] >= GAP_P75 else "consistent_fade"
    if row["users_90d"] >= SCALE_P90 and 0 < row["avg_position"] <= 10:
        return "large_page_blindspot"
    return "stable_low_priority"

queue["archetype"] = queue.apply(classify, axis=1)
print(queue["archetype"].value_counts())


archetype
stable_low_priority          6392
large_page_blindspot          523
visibility_without_clicks     119
consistent_fade                81
Name: count, dtype: int64


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("docs/img", exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

# Chart 1 — precision@K, model vs baseline
ks = ["20", "50", "100", "200"]
baseline_vals = results_table["baseline_precision@K"].tolist()
logreg_vals = results_table["logreg_precision@K"].tolist()
rf_vals = results_table["rf_precision@K"].tolist()
x = np.arange(len(ks)); w = 0.25
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(x - w, baseline_vals, w, label="Baseline (hand rule)", color="#B0B0B0")
ax.bar(x, logreg_vals, w, label="Logistic Regression", color="#4C72B0")
ax.bar(x + w, rf_vals, w, label="Random Forest", color="#DD8452")
ax.axhline(0.517, color="black", linestyle="--", linewidth=1, label="Base rate (0.517)")
ax.set_xticks(x); ax.set_xticklabels([f"K={k}" for k in ks])
ax.set_ylabel("Precision@K"); ax.set_title("Precision@K: model vs baseline, same client-held-out split")
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig("docs/img/precision_at_k.png", dpi=150); plt.close()

# Chart 2 — top features
feat_imp = pd.DataFrame({"feature": X.columns, "imp": permutation_importance(
    logreg, X_test_s, y_test, n_repeats=10, random_state=RANDOM_SEED, scoring="roc_auc"
).importances_mean}).sort_values("imp", ascending=False).head(5)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.barh(feat_imp["feature"][::-1], feat_imp["imp"][::-1], color="#4C72B0")
ax.set_xlabel("Permutation importance (drop in ROC AUC)"); ax.set_title("What the model leans on")
plt.tight_layout(); plt.savefig("docs/img/top_features.png", dpi=150); plt.close()

# Chart 3 — archetype counts
counts = queue["archetype"].value_counts()
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.barh(counts.index, counts.values, color="#4C72B0")
for i, v in enumerate(counts.values):
    ax.text(v + 60, i, str(v), va="center")
ax.set_xlabel(f"Number of pages (n={len(queue)})"); ax.set_title("Action queue: pages by archetype")
plt.tight_layout(); plt.savefig("docs/img/archetype_counts.png", dpi=150); plt.close()

print("wrote docs/img/precision_at_k.png, top_features.png, archetype_counts.png")


wrote docs/img/precision_at_k.png, top_features.png, archetype_counts.png


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 8. Demo outline — Week-8 showcase (5 minutes, optional)

**Question (30 sec):** Among FlyRank's anonymized content-performance data, which pre-decision signals are associated with a page's visibility declining — and is that association strong enough to beat a transparent hand rule at building a reviewer's queue?

**Method (1 min):** Logistic Regression and Random Forest, trained on 23 current-window features (no label leakage, no IDs) from 30,000 pseudonymized pages across 32 clients. Validated with a **client-held-out split** — never mixing one client's rows across train and test — and scored against a staleness-and-CTR-gap baseline on that exact same split.

**One chart (1.5 min):** *Precision@K: model vs baseline, same client-held-out split.* Logistic Regression beats the baseline at every K (0.85 → 0.705 as K grows from 20 to 200), while Random Forest — despite a higher ROC AUC — tracks the baseline almost exactly.

**One honest result (1 min):** *Why the split choice matters.* The same Random Forest model scores 0.693 ROC AUC on a naive random split and only 0.589 on the honest client-held-out split — a 0.10 gap, and 0.589 is the number to trust. A higher AUC does not automatically mean a better ranked queue; precision@K is the metric this decision actually needs.

**One recommendation (1 min):** Route the top 200 pages by `pred_proba` into a capped action queue (precision@200 = 0.705), split into four archetypes so the reviewer's first move differs by page: `consistent_fade` → refresh content; `visibility_without_clicks` → check title/meta/snippet; `large_page_blindspot` → prioritize for reach; `stable_low_priority` → leave alone this cycle. This is decision-support that orders an existing human review queue — not a forecast, and not a replacement for that reviewer's judgment on any single page.

## 9. Two shareable cuts

### Social post (LinkedIn)

Wrapped up my ML capstone at FlyRank, and the most useful thing I learned wasn't a modeling trick — it was a validation trap.

I built a classifier to help prioritize which web pages a content team should review first, using an anonymized dataset of 30,000 pages across 32 clients. Early on, a Random Forest model looked great: 0.693 ROC AUC. Strong signal, right?

Then I switched from a random train/test split to a **client-held-out split** — making sure no client's pages ever crossed between train and test — and that same model dropped to 0.589. Same data, same features, same model. The only thing that changed was whether the split was honest about how this model will actually be used: on clients it has never seen before.

The bigger lesson: a higher AUC didn't even mean a better model for the actual decision. On precision@K — "of the top K flagged pages, how many are really declining" — a simpler Logistic Regression beat both the Random Forest and a hand-written baseline rule at every K tested, despite having a lower AUC. The metric that matches the decision beats the metric that looks better on a slide.

Full write-up (question, method, honest limitations, and the resulting action queue) is linked below. #MachineLearning #DataScience #MLOps

### Employer-facing summary (3 sentences)

I built a Logistic Regression and Random Forest classifier to rank web pages by likelihood of visibility decline, trained on FlyRank's anonymized content-performance data (30,000 pages, 32 clients, 23 leakage-checked features), validated with a client-held-out split to reflect real deployment conditions. The model showed a modest but genuine lift over a transparent hand-written baseline — Logistic Regression reached 0.705 precision at the top 200 ranked pages, versus a 0.517 base rate — while a client-held-out AUC of 0.589 (vs. an inflated 0.693 on a naive random split) kept the result honest about its limits. The output is a capped, archetype-tagged action queue designed as decision-support for a human reviewer, not an autonomous forecast.